# Combler les heures manquantes avec les capteurs voisins

**Problème** : nos 3 arcs ont des heures sans mesure (capteur en panne, arc « Invalide »...).
Certaines années, c'est énorme : 72 % pour Convention en 2024, 100 % pour Champs-Élysées en 2026.

**Idée (de Rim)** : les capteurs des rues d'à côté mesurent le trafic au même moment, et leur trafic ressemble au nôtre.
On apprend la relation *« débit des voisins → notre débit »* sur les heures où tout est mesuré,
puis on l'utilise pour deviner les heures où il nous manque.

**Question de ce notebook** : est-ce vraiment mieux que des méthodes simples (ffill, interpolation linéaire...) ?

Plan :
1. Charger notre débit (la **cible**)
2. Charger les débits des capteurs des rues voisines
3. Choisir les capteurs utiles (corrélation)
4. Comparer les méthodes sur des **faux trous** (on cache des heures qu'on connaît)
5. Regarder la durée de nos **vrais** trous
6. Combler les trous et sauvegarder

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor

from dataset_create import PROCESSED_DIR, charger
from imputation_voisins import VOISINS, voisins

SEUIL_CORR = 0.8  # on garde les capteurs dont la corrélation avec notre débit est >= ce seuil
PART_TEST = 0.2   # on cache 20 % des heures connues pour tester

## 1. Notre débit (la cible)

`charger()` donne une ligne par axe et par heure, avec `debit` = NaN quand il n'y a pas de mesure fiable
(capteur en panne, arc « Invalide »...). On en fait un tableau : **une ligne = une heure**, **une colonne = un axe**.

In [ ]:
df = charger()

cible = df.pivot(index="datetime", columns="axe", values="debit")
cible.index = cible.index.tz_convert("UTC")  # les capteurs voisins sont en heure UTC
cible = cible.asfreq("h")  # grille horaire complète, sans heure sautée
cible.tail()

In [ ]:
# Pourcentage d'heures sans mesure, par année
annee = cible.index.year
(cible.isna().groupby(annee).mean() * 100).round(1)

## 2. Les débits des capteurs voisins

`voisins()` (dans `imputation_voisins.py`) renvoie le même genre de tableau : une ligne par heure,
une colonne par capteur, nommée `<rue>_<numéro d'arc>`. Les rues utilisées sont dans `VOISINS`.

La première fois ça lit toutes les archives (~10 min), ensuite c'est lu depuis `data/external/voisins.csv`.

In [ ]:
print(VOISINS)

capteurs = voisins()
capteurs = capteurs.reindex(cible.index)  # mêmes heures que la cible
print(capteurs.shape[1], "capteurs dans les rues voisines")

## 3. Choisir les capteurs utiles

Pour chaque axe, un capteur est gardé si :
- il est dans une des **rues voisines** de l'axe ;
- il est **corrélé** à notre débit (corrélation >= `SEUIL_CORR`) ;
- il est **mesuré pendant nos trous** (au moins 24 heures où notre débit manque).

La dernière règle est importante : certains « voisins » sont en fait **des copies de notre arc** (même capteur,
autre numéro). Corrélation = 1, mais ils tombent en panne en même temps que nous : ils n'aident jamais à combler
un trou, et ils font croire au modèle qu'il est parfait (c'était le RMSE ~0 de Rim sur Convention).

In [ ]:
capteurs_choisis = {}

for axe in cible.columns:
    debit = cible[axe]
    colonnes = [c for c in capteurs.columns if c.rsplit("_", 1)[0] in VOISINS[axe]]

    correlation = capteurs[colonnes].corrwith(debit, min_periods=500)  # au moins 500 heures en commun
    mesure_pendant_trous = capteurs.loc[debit.isna(), colonnes].notna().sum()

    garde = (correlation >= SEUIL_CORR) & (mesure_pendant_trous >= 24)
    capteurs_choisis[axe] = list(correlation[garde].sort_values(ascending=False).index)

    print(f"{axe} : {garde.sum()} capteurs gardés, les 5 meilleurs :")
    print(correlation[garde].sort_values(ascending=False).head().round(3).to_string(), "\n")

In [ ]:
def features(axe):
    """X = débits des capteurs choisis + heure + jour de la semaine (heure de Paris)."""
    X = capteurs[capteurs_choisis[axe]].copy()
    heure_paris = X.index.tz_convert("Europe/Paris")
    X["heure"] = heure_paris.hour
    X["jour_sem"] = heure_paris.day_of_week
    return X


def nouveau_modele():
    # HistGradientBoosting (même famille que LightGBM) : accepte les NaN dans X,
    # donc si un voisin est aussi en panne, le modèle fait avec les autres.
    return HistGradientBoostingRegressor(max_iter=500, learning_rate=0.05)

## 4. Comparer les méthodes sur des faux trous

On ne peut mesurer une erreur que sur des heures dont on **connaît** le vrai débit. Donc on fabrique des **faux trous** :
on cache 20 % des heures connues, chaque méthode les devine, et on compare au vrai débit.

**La durée des trous change tout** : l'interpolation est excellente pour 1 heure manquante (le trafic change peu
d'une heure à l'autre), mais pour une semaine entière elle trace juste une ligne droite. On teste donc des trous de
**1 h, 6 h, 24 h et 1 semaine**.

Les méthodes comparées :
| Méthode | Principe |
|---|---|
| `ffill` | on recopie la dernière valeur connue |
| `interpolation` | ligne droite entre la valeur d'avant et celle d'après le trou |
| `profil moyen` | moyenne du débit pour cette heure et ce jour de la semaine (calculée sur le train) |
| `voisins` | le boosting sur les capteurs voisins |

Seules les heures où au moins un voisin est mesuré sont utilisées, pour que toutes les méthodes soient testées sur les mêmes heures.

In [ ]:
def faux_trous(utilisable, duree, seed=0):
    """Cache au hasard des trous de `duree` heures, sur environ PART_TEST des heures utilisables."""
    numero_bloc = np.arange(len(utilisable)) // duree            # on découpe le temps en blocs de `duree` heures
    bloc_complet = utilisable.groupby(numero_bloc).all()           # blocs où tout est utilisable
    candidats = bloc_complet.index[bloc_complet]

    rng = np.random.default_rng(seed)                              # tirage fixe = reproductible
    blocs_caches = rng.choice(candidats, size=int(PART_TEST * len(candidats)), replace=False)

    test = pd.Series(np.isin(numero_bloc, blocs_caches), index=utilisable.index)
    train = utilisable & ~test
    return train, test

In [ ]:
def predire_toutes_methodes(axe, train, test):
    X = features(axe)
    y = cible[axe]
    y_cache = y.where(~test)  # le débit avec les faux trous (NaN)

    predictions = {}
    predictions["ffill"] = y_cache.ffill()[test]
    predictions["interpolation"] = y_cache.interpolate()[test]

    profil = y[train].groupby([X["jour_sem"][train], X["heure"][train]]).mean()
    cles_test = pd.MultiIndex.from_arrays([X["jour_sem"][test], X["heure"][test]])
    predictions["profil moyen"] = pd.Series(profil.reindex(cles_test).to_numpy(), index=y.index[test])

    modele = nouveau_modele().fit(X[train], y[train])
    predictions["voisins"] = pd.Series(modele.predict(X[test]).clip(0), index=y.index[test])
    return predictions


resultats = []
for axe in cible.columns:
    y = cible[axe]
    utilisable = y.notna() & capteurs[capteurs_choisis[axe]].notna().any(axis=1)

    for duree in [1, 6, 24, 168]:
        train, test = faux_trous(utilisable, duree)
        vrai = y[test]
        for methode, prediction in predire_toutes_methodes(axe, train, test).items():
            mae = (prediction - vrai).abs().mean()
            resultats.append({"axe": axe, "duree_trou_h": duree, "methode": methode,
                              "mae_%": round(100 * mae / vrai.mean(), 1)})

resultats = pd.DataFrame(resultats)
resultats.pivot_table(index=["axe", "duree_trou_h"], columns="methode", values="mae_%")

**Lecture** : erreur moyenne en % du débit moyen (plus c'est bas, mieux c'est), selon la durée du trou.

## 5. Et nos vrais trous, ils durent combien ?

Pour savoir quelle méthode compte vraiment, on regarde comment les heures manquantes se répartissent
selon la durée du trou dont elles font partie.

In [ ]:
repartition = {}
for axe in cible.columns:
    manque = cible[axe].isna()
    numero_trou = (manque != manque.shift()).cumsum()          # change de numéro à chaque début/fin de trou
    duree_du_trou = manque.groupby(numero_trou).transform("size")[manque]

    categories = pd.cut(duree_du_trou, [0, 1, 6, 24, 168, np.inf],
                        labels=["1 h", "2-6 h", "7-24 h", "1-7 jours", "> 1 semaine"])
    repartition[axe] = categories.value_counts(normalize=True).sort_index() * 100

print("% des heures manquantes, selon la durée du trou :")
pd.DataFrame(repartition).round(1)

## 6. Combler les trous

On réentraîne le modèle voisins sur **toutes** les heures connues, puis on prédit les heures où :
- notre débit **manque**,
- et au moins un capteur choisi **est mesuré** (sinon on n'a aucune info, on laisse vide).

Les vraies mesures ne sont jamais modifiées. La colonne `impute` dit si une valeur est devinée.

In [ ]:
debit_complete = cible.copy()

for axe in cible.columns:
    X = features(axe)
    y = cible[axe]
    un_voisin_mesure = X[capteurs_choisis[axe]].notna().any(axis=1)

    connu = y.notna() & un_voisin_mesure
    modele = nouveau_modele().fit(X[connu], y[connu])

    a_combler = y.isna() & un_voisin_mesure
    debit_complete.loc[a_combler, axe] = modele.predict(X[a_combler]).clip(0)

# % d'heures sans mesure, avant / après
pd.concat({"avant": cible.isna().groupby(annee).mean() * 100,
           "après": debit_complete.isna().groupby(annee).mean() * 100}, axis=1).round(1)

In [ ]:
# On remet au format "une ligne = un axe x une heure", comme les autres fichiers
morceaux = []
for axe in cible.columns:
    morceaux.append(pd.DataFrame({
        "datetime_utc": cible.index,
        "axe": axe,
        "debit_voisins": debit_complete[axe].to_numpy(),
        "impute": (cible[axe].isna() & debit_complete[axe].notna()).to_numpy(),
    }))
resultat = pd.concat(morceaux, ignore_index=True)

resultat.to_csv(PROCESSED_DIR / "debit_voisins.csv", index=False)
resultat.head()